In [ ]:
%matplotlib qt5
import pygad
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import mplcyberpunk #Dodatkowy styl dla wykresów - efekt neonowy

#1. Konfiguracja problemu
N =10 #liczba zadań i wykonawców
SEED = 42 #ustawienie ziarna losowości dla powtarzalności wyników

np.random.seed(SEED)  
cost_matrix = np.random.randint(1, 51, size=(N, N))

print("Macierz kosztów k(i,j):")
print(cost_matrix)

#2. Funkacja dopasowania (fitness function)
def fitness_func(ga_instance, solution, solution_idx):
    total_cost = 0
    for task_idx, worker_idx in enumerate(solution):
        total_cost += cost_matrix[task_idx, int(worker_idx)]

    #fitnes rośnie, gdy koszt maleje 
    fitness = 1.0 / total_cost if total_cost != 0 else 0
    return fitness

# 3. Konfiguracja algorytmu genetycznego
ga_instance = pygad.GA(num_generations=200, 
                       num_parents_mating=10, 
                       fitness_func=fitness_func, 
                       sol_per_pop=50, 
                       num_genes=N, 
                       gene_space=list(range(N)), 
                       gene_type=int,
                       parent_selection_type="sss", 
                       keep_parents=2, 
                       crossover_type="single_point", 
                       mutation_type="swap", 
                       mutation_probability=0.1,
                       allow_duplicate_genes=False
)

# 4. Uruchomienie algorytmu genetycznego
ga_instance.run()

# 5. Podsumowanie wyników
solution, solution_fitness, solution_idx = ga_instance.best_solution()
best_cost = 1.0 / solution_fitness 

print("-" * 30)
print(f"Najlepsze rozwiązanie (przypisanie): {solution}")
print(f"Minimalny koszt: {best_cost}")

# 6. Trzy wykresy 
#Usdtaienie stylu dark mode dla wykresów
plt.style.use('cyberpunk')
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 8))

# Wykres 1: Fitness (zbieżność)
fitness_history = ga_instance.best_solutions_fitness
ax1.plot(fitness_history, marker='o', markevery=20, markersize=4) #  markery co 20 pokoleń
ax1.set_title("1. Zbieżność algorytmu genetycznego (Fitness)", fontsize=14, pad=15)
ax1.set_xlabel("Pokolenie")
ax1.set_ylabel("Wartość Fitness")
mplcyberpunk.make_lines_glow(ax1) # Dodanie efektu glow do linii wykresu

# Wykres 2: Spadek realnego kosztu 
# Wyciagamy historię fitness i zmieniamy ja na koszt
cost_history = [1.0 / f if f != 0 else 0 for f in fitness_history]
ax2.fill_between(range(len(cost_history)), cost_history, color="#ff0055", alpha=0.2) #  Cieniowanie pod linią
ax2.plot(cost_history, color="#ff0055", linewidth=2)
ax2.set_title("2. Spadek kosztu całkowitego", fontsize=14, color="#ff0055")
ax2.set_xlabel("Pokolenie")
ax2.set_ylabel("Suma kosztów (k)")




# Wykres 3: Wizualizacja macierzy kosztów
im = ax3.imshow(cost_matrix, cmap='magma', interpolation='nearest')
fig.colorbar(im, ax=ax3, label='Koszt k(i,j)')

for task_idx, worker_idx in enumerate(solution):
    # Rysujemy X i ramke dla wybranego przydziału 
    ax3.scatter(worker_idx, task_idx, s=150, color='cyan', edgecolors='white', zorder=5)
    #podswietlona ramka
    rect = patches.Rectangle((worker_idx - 0.4, task_idx - 0.4), 0.8, 0.8, linewidth=2, edgecolor='cyan', facecolor='none', alpha=0.8)
    ax3.add_patch(rect)

ax3.set_title("3. Optymalny przydział (Mapa połączeń)", fontsize=14,color='cyan', pad=20)
ax3.set_xticks(range(N))
ax3.set_yticks(range(N))
ax3.set_xlabel("Wykonawcy (j)")
ax3.set_ylabel("Zadania (i)")

#Podswietlenie krawedzi wykresow
mplcyberpunk.add_underglow(ax1)
mplcyberpunk.add_underglow(ax2)


plt.tight_layout(pad=4.0)  #Dostosowanie układu, aby tytuły się nie nakładały
print("-" * 30)
print(f"Najlepsze rozwiązanie (przypisanie): {solution}")
print(f"Minimalny koszt: {best_cost}")
plt.show()

Macierz kosztów k(i,j):
[[39 29 15 43  8 21 39 19 23 11]
 [11 24 36 40 24  3 22  2 24 44]
 [30 38  2 21 33 12 22 44 25 49]
 [27 42 28 16 15 47 44  3 37  7]
 [21  9 39 18  4 25 14 50  9 26]
 [ 2 20 28 47  7 44  8 47 35 14]
 [17 36 50 40  4  2  6 42  4 29]
 [18 26 44 34 10 36 14 31 48 15]
 [ 8 14 23 40 21 16 45 18 47 24]
 [26 25 45 41 29 15 45  1 25  7]]
------------------------------
Najlepsze rozwiązanie (przypisanie): [9 5 2 3 8 4 6 0 1 7]
Minimalny koszt: 87.0
